# Sea ice volume budget

In [2]:
## import required packages
import numpy as np
import pandas as pd
import sys
import xarray as xr
import matplotlib.pyplot as plt
import s3fs
import cmocean
import gsw
import warnings
warnings.filterwarnings("ignore", module='distributed')

In [3]:
# initialize s3 filesystem
s3_options = dict(anon=False)

In [4]:
# open geometry file for HH field
HH_grid = xr.open_dataset("~/efs-mount-point/mzahn/sassie/HH/GRID/GRID_GEOMETRY_SASSIE_HH_V1R1_NATIVE_LLC1080.nc")

In [5]:
# function to open zarr store with a provided s3 bucket path
def open_zarr_store(s3_path, s3_options):
    # initalize s3 file system
    s3 = s3fs.S3FileSystem(**s3_options)

    # define location of zarr store and open
    store = s3fs.S3Map(root=s3_path, s3=s3, check=False)
    zarr_store = xr.open_zarr(store)
    
    return zarr_store

In [6]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:33447")
client

<Client: 'tcp://127.0.0.1:33447' processes=8 threads=32, memory=123.95 GiB>

## Budget expression

**Mean sea-ice thickness**, $h$

*Model variable:* `HEFF`

$$
\frac{\partial h}{\partial t} = \left. \frac{\partial h}{\partial t} \right|_{\text{dynamics}} + \left. \frac{\partial h}{\partial t} \right|_{\text{thermo}} + \left. \frac{\partial h}{\partial t} \right|_{\text{snow-flooding}}
$$

**Dynamics**

$$
\left. \frac{\partial h}{\partial t} \right|_{\text{dynamics}} = \left. \frac{\partial h}{\partial t} \right|_{\text{advection}} + \left. \frac{\partial h}{\partial t} \right|_{\text{diffusion}}
$$

$$
\left. \frac{\partial h}{\partial t} \right|_{\text{dynamics}} = \underbrace{\nabla \cdot (\vec{u}_{\text{ice}}\, h)}_{\text{advection}} + \underbrace{\kappa \nabla^2 h}_{\text{diffusion}}
$$

Advection terms: `ADVxHEFF`, `ADVyHEFF`  
Diffusion terms: `DFxEHEFF`, `DFyEHEFF`

---

**Thermodynamics**

$$
\left. \frac{\partial h}{\partial t} \right|_{\text{thermo}} = \left. \frac{\partial h}{\partial t} \right|_{\text{ocean-atm}} + \left. \frac{\partial h}{\partial t} \right|_{\text{atm-ice}} + \left. \frac{\partial h}{\partial t} \right|_{\text{ocean-ice}}
$$

- $\left. \frac{\partial h}{\partial t} \right|_{\text{ocean-ice}}$: `SIdHbOCN` – *HEFF rate of change by ocean-ice flux*  
- $\left. \frac{\partial h}{\partial t} \right|_{\text{ocean-atm}}$: `SIdHbATO` – *HEFF rate of change by open ocean-atm flux*  
- $\left. \frac{\partial h}{\partial t} \right|_{\text{atm-ice}}$: `SIdHbATC` – *HEFF rate of change by atm flux over sea ice*

---

**Snow Flooding**

- $\left. \frac{\partial h}{\partial t} \right|_{\text{snow-flooding}}$: `SIdHbFLO` – *HEFF rate of change by flooding snow*

### Open daily mean ZARR datasets

In [6]:
# open SIheff zarr store
SIheff_da = open_zarr_store('s3://ecco-processed-data/SASSIE/N1/HH/ZARR/SIheff_AVG_DAILY.ZARR/', s3_options).SIheff

# open THETA zarr store - [degC]
theta_da = open_zarr_store('s3://ecco-processed-data/SASSIE/N1/HH/ZARR/THETA_AVG_DAILY.ZARR/', s3_options).THETA

# open SIarea zarr store - [fraction of sea ice, 0 to 1]
ice_da = open_zarr_store('s3://ecco-processed-data/SASSIE/N1/HH/ZARR/SIarea_AVG_DAILY.ZARR/', s3_options).SIarea*100

In [7]:
# carve out beaufort region
salt_beaufort = salt_da.isel(i=slice(450,750),j=slice(110,390))
theta_beaufort = theta_da.isel(i=slice(450,750),j=slice(110,390))
ice_beaufort = ice_da.isel(i=slice(450,750),j=slice(110,390))

HH_grid_land = HH_grid.maskC.isel(k=0).where(HH_grid.maskC.isel(k=0)==0)
HH_land_beaufort = HH_grid_land.isel(i=slice(450,750),j=slice(110,390))